# Lab 3.3 &mdash; Build a StateGraph From Scratch

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; Memory, State &amp; the LangGraph Substrate**

### What you'll do
- Implement nodes, edges, conditional edges and cycles in ~40 lines
- Make state an explicit object every node reads and writes
- Add a step budget so a cycle terminates
- Run the same graph on real LangGraph and compare

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The substrate, built before it is used.** You will understand LangGraph better for
> having written the 40 lines it is hiding, and every later module sits on this.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# These are the tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

A StateGraph is four things and no more:

| Piece | What it is |
|---|---|
| **State** | a dict with a declared shape, threaded through everything |
| **Node** | a function: state in, *partial* state out |
| **Edge** | what runs next, always |
| **Conditional edge** | what runs next, given the state |

A **cycle** is just an edge that points backwards. Build all of it here, in plain Python.

## Section 1 &mdash; Nodes return partial state

The single most important convention: a node returns **only the keys it changed**, and the engine
merges. That is what makes nodes composable and independently testable.

In [ ]:
def read_ledger(state: dict) -> dict:
    """Node: look up the payment. Returns ONLY the keys it changed."""
    obs = lookup_payment(state["ref"])
    return {"findings": [f"ledger: {obs}"], "steps": state["steps"] + 1}

def read_policy(state: dict) -> dict:
    """Node: look up the policy for whatever reason code the ledger gave."""
    rec = LEDGER.get(state["ref"], {})
    obs = policy_for(rec.get("reason_code")) if rec.get("reason_code") else "no reason code"
    needs_human = rec.get("reason_code") in NEEDS_HUMAN
    return BLANK                     # TODO: the three keys this node changes --
                                     # findings (a one-item list), needs_human, and steps

In [ ]:
# --- Self-check: Section 1
_s0 = {"ref": "PMT-1005", "findings": [], "needs_human": False, "steps": 0, "answer": None}

check("a node returns only what it changed",
      lambda: set(read_ledger(_s0)) == {"findings", "steps"},
      "returning the whole state makes nodes impossible to compose")
check("read_ledger records its observation",
      lambda: "SANCTIONS_REVIEW" in read_ledger(_s0)["findings"][0])
check("read_policy returns its three keys",
      lambda: set(read_policy(_s0)) == {"findings", "needs_human", "steps"})
check("a sanctions hold is flagged for a human",
      lambda: read_policy(_s0)["needs_human"] is True)
check("an insufficient-funds case is not",
      lambda: read_policy({**_s0, "ref": "PMT-1002"})["needs_human"] is False)
check("nodes do not mutate the state they were given",
      lambda: (read_ledger(_s0), _s0["steps"] == 0)[1],
      "return a new dict; never edit state in place")

## Section 2 &mdash; Reducers: how partial updates merge

`steps` should be replaced by the new value. `findings` should **accumulate**. That difference is
the reducer, declared once per key rather than remembered in every node.

In [ ]:
def replace(old, new):
    return new

def append(old, new):
    return list(old) + list(new)

REDUCERS = {
    "findings": append,              # every node's findings are kept, in order
    "steps": replace,
    "needs_human": replace,
    "answer": replace,
    "ref": replace,
}

def merge(state: dict, update: dict) -> dict:
    """Apply a node's partial update using the declared reducer for each key."""
    out = dict(state)
    for key, value in update.items():
        reducer = REDUCERS.get(key, replace)
        out[key] = BLANK             # TODO: combine the old value with the new one
    return out

In [ ]:
# --- Self-check: Section 2   (fixtures built lazily -- a module-level call into an
#                              unfilled function would crash the cell instead of printing [TODO])
def _a():
    return merge(_s0, {"findings": ["one"], "steps": 1})

def _b():
    return merge(_a(), {"findings": ["two"], "steps": 2})

check("findings accumulate across nodes", lambda: _b()["findings"] == ["one", "two"],
      "this is the append reducer doing its job")
check("steps is replaced, not appended", lambda: _b()["steps"] == 2)
check("untouched keys survive the merge", lambda: _b()["ref"] == "PMT-1005")
check("the original state is not mutated", lambda: _s0["findings"] == [])
check("an undeclared key defaults to replace",
      lambda: merge(_s0, {"novel": 7})["novel"] == 7)
check("two parallel writes to findings both survive",
      lambda: merge(merge(_s0, {"findings": ["x"]}), {"findings": ["y"]})["findings"] == ["x", "y"],
      "with the default replace reducer, one of these would silently vanish")

## Section 3 &mdash; The engine: edges, conditions and a cycle

Forty lines. Nodes run, updates merge, and a conditional edge decides what comes next &mdash; including
going backwards.

In [ ]:
END = "__end__"

class Graph:
    def __init__(self, reducers):
        self.nodes, self.edges, self.conditions = {}, {}, {}
        self.entry = None
        self.reducers = reducers

    def add_node(self, name, fn):
        self.nodes[name] = fn
        return self

    def add_edge(self, src, dst):
        self.edges[src] = dst
        return self

    def add_conditional_edge(self, src, fn):
        """fn(state) -> the name of the next node, or END."""
        self.conditions[src] = fn
        return self

    def set_entry(self, name):
        self.entry = name
        return self

    def run(self, state, max_steps=8):
        """Execute until END or the budget is spent. Returns (final_state, path)."""
        current, path = self.entry, []
        while current != END:
            if state["steps"] >= max_steps:
                return merge(state, {"answer": "stopped: step budget"}), path
            path.append(current)
            update = self.nodes[current](state)
            state = merge(state, update)
            if current in self.conditions:
                current = self.conditions[current](state)
            else:
                current = BLANK      # TODO: the plain edge out of this node, or END if there is none
        return state, path

In [ ]:
# --- Self-check: Section 3
def write_note(state):
    verdict = "human decision required" if state["needs_human"] else "operations may proceed"
    return {"answer": f"{state['ref']}: {verdict}", "steps": state["steps"] + 1}

def enough(state):
    """Conditional edge: two findings is enough to conclude; otherwise go round again."""
    return "write_note" if len(state["findings"]) >= 2 else "read_ledger"

def build():
    return (Graph(REDUCERS)
            .add_node("read_ledger", read_ledger)
            .add_node("read_policy", read_policy)
            .add_node("write_note", write_note)
            .add_edge("read_ledger", "read_policy")
            .add_conditional_edge("read_policy", enough)
            .add_edge("write_note", END)
            .set_entry("read_ledger"))

def _run(ref="PMT-1005"):
    return build().run({"ref": ref, "findings": [], "needs_human": False,
                        "steps": 0, "answer": None})

check("the graph reaches an answer", lambda: _run()[0]["answer"] is not None)
check("it visits the nodes in order",
      lambda: _run()[1][:3] == ["read_ledger", "read_policy", "write_note"])
check("findings accumulated from both reader nodes",
      lambda: len(_run()[0]["findings"]) == 2)
check("a sanctions hold ends with a human decision",
      lambda: "human decision required" in _run("PMT-1005")[0]["answer"])
check("an insufficient-funds case does not",
      lambda: "operations may proceed" in _run("PMT-1002")[0]["answer"])
check("a graph that never satisfies its condition stops on the budget",
      lambda: "step budget" in (Graph(REDUCERS)
              .add_node("read_ledger", read_ledger)
              .add_conditional_edge("read_ledger", lambda s: "read_ledger")
              .set_entry("read_ledger")
              .run({"ref": "PMT-1002", "findings": [], "needs_human": False,
                    "steps": 0, "answer": None})[0]["answer"]),
      "a cycle without a budget is an infinite loop")

try:
    final, path = _run()
    print("path:", " -> ".join(path))
    print("state:", json.dumps({k: v for k, v in final.items() if k != "findings"}, indent=1))
    for f in final["findings"]:
        print("  -", f[:90])
except NameError:
    print("(fill in the blanks above, then re-run)")

## Run it for real

The same graph on actual LangGraph. Note how little changes: `TypedDict` instead of a plain dict,
`Annotated[list, add]` instead of your `REDUCERS` table, and `add_conditional_edges` instead of
your dictionary of functions.

In [ ]:
try:
    from typing import Annotated
    from typing_extensions import TypedDict
    from operator import add
    from langgraph.graph import StateGraph, START, END as LG_END

    class CaseState(TypedDict):
        ref: str
        findings: Annotated[list, add]        # <- your `append` reducer, declared
        needs_human: bool
        steps: int
        answer: str | None

    g = StateGraph(CaseState)
    g.add_node("read_ledger", read_ledger)
    g.add_node("read_policy", read_policy)
    g.add_node("write_note", write_note)
    g.add_edge(START, "read_ledger")
    g.add_edge("read_ledger", "read_policy")
    g.add_conditional_edges("read_policy", enough,
                            {"write_note": "write_note", "read_ledger": "read_ledger"})
    g.add_edge("write_note", LG_END)
    app = g.compile()

    out = app.invoke({"ref": "PMT-1005", "findings": [], "needs_human": False,
                      "steps": 0, "answer": None})
    print("answer  :", out["answer"])
    print("findings:", len(out["findings"]))
    print("\nSame nodes, same conditional edge, same reducer. The engine is the part you wrote.")
except ImportError as exc:
    print(f"LangGraph not importable here ({exc}). The graded cells above do not need it.")
except NameError:
    print("(fill in the blanks above, then re-run this cell)")
except Exception as exc:
    print(f"<graph run failed: {type(exc).__name__}: {exc}>")

### Read it

Your `merge` is LangGraph's reducer machinery. Your `conditions` dict is `add_conditional_edges`.
Your `max_steps` guard is its recursion limit. What LangGraph adds on top is the part that is
genuinely hard: **checkpointing** &mdash; and that is the next lab.

In [ ]:
score()

## Your turn

1. Add a `read_counterparty` node and run it **in parallel** with `read_policy`. What must be true
   of the `findings` reducer for both results to survive? You already know &mdash; now prove it.
2. `run()` counts steps from the state, so a node that forgets to increment `steps` makes the
   budget unenforceable. Move the counting into the engine. What does that cost you in node
   independence?